In [1]:
import os
import torch
import random
import numpy as np
import pandas as pd
import re
from datasets import load_from_disk, DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import DataCollatorForCompletionOnlyLM

In [2]:
def set_seed(random_seed):
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)

set_seed(42)

In [3]:
data_path = "../../data/persona_data/empathy_multiturn_qa_dataset.csv" # 토큰화 확인을 위한 데이터 호출.
df = pd.read_csv(data_path, usecols=['situation', 'context', 'response'])

In [4]:
df.head()

,situation,context,response
0,청소하시는 아주머니가 아파트 계단 청소를 항상 깔끔하게 한다.,A: 우리 아파트 계단이 요즘 너무 깨끗해졌어.\nB: 아 진짜? 너 지난번에 너희...,너무 좋은 일이다. 아파트가 머지않아 반짝반짝해지겠는걸?
1,청소하시는 아주머니가 아파트 계단 청소를 항상 깔끔하게 한다.,A: 그랬었지. 근데 이번에 새로 청소하시는 아주머니께서 오셨는데 너무 깔끔하고 꼼...,공동 주택 생활을 하려면 이웃끼리 배려하는 맘도 중요해. 하지만 보이지 않는 곳에서...
2,청소하시는 아주머니가 아파트 계단 청소를 항상 깔끔하게 한다.,A: 그러니까 옆집 주민도 나와 같은 생각인가 봐. 우연히 만났는데 청소 아주머니를...,아침에 집을 나설 때부터 기분이 좋아지면 하루가 다 즐거울 것 같아.
3,청소하시는 아주머니가 아파트 계단 청소를 항상 깔끔하게 한다.,A: 맞아. 예전에는 그냥 좀 지저분하다고만 느꼈었거든. 그런데 막상 깔끔하게 청소...,네 말을 들으니 나도 기분이 막 좋아지는 것 같다. 그 아주머니에게 나도 감사하다고...
4,청소하시는 아주머니가 아파트 계단 청소를 항상 깔끔하게 한다.,A: 맞아. 하루를 이렇게 상쾌하게 시작하는구나 생각하니 그 아주머니가 더 고맙더라...,우리는 살면서 감사할 일들이 참 많은 것 같아. 그런데 이런 생각을 하는 너도 참 ...


In [5]:
for col in df.columns:
    df[col] = df[col].str.replace('\n', ' ', regex=False)
    df[col] = df[col].str.replace('\r', ' ', regex=False)

In [6]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


PyTorch 버전: 2.10.0+cu128
CUDA 사용 가능: True
CUDA 버전: 12.8
GPU 이름: Tesla V100-SXM2-32GB
GPU 메모리: 31.74 GB


In [7]:
MODEL_NAME = "LGAI-EXAONE/EXAONE-4.0-1.2B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,slow_tokenizer=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
print(tokenizer.chat_template)

In [8]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)  # 90% 훈련, 10% 테스트
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['situation', 'context', 'response'],
        num_rows: 38342
    })
    test: Dataset({
        features: ['situation', 'context', 'response'],
        num_rows: 4261
    })
})


In [9]:
if isinstance(dataset, DatasetDict):
    print("데이터셋 구조:")
    for split, ds in dataset.items():
        print(f"  - {split}: {len(ds)} 샘플")
else:
    print(f"총 샘플: {len(dataset)}")

데이터셋 구조:
  - train: 38342 샘플
  - test: 4261 샘플


In [10]:
sample = dataset['train'][0] if isinstance(dataset, DatasetDict) else dataset[0]

In [11]:
def random_sample_dataset(dataset, train_size=5000,test_size=500, seed=42):
    """랜덤 샘플링"""
    random.seed(seed)
    
    if isinstance(dataset, DatasetDict):
        sampled = DatasetDict()
        
        if 'train' in dataset:
            indices = random.sample(range(len(dataset['train'])), min(train_size, len(dataset['train'])))
            sampled['train'] = dataset['train'].select(indices)
        
        if 'test' in dataset:
            indices = random.sample(range(len(dataset['test'])), min(test_size, len(dataset['test'])))
            sampled['test'] = dataset['test'].select(indices)
        
        return sampled
    else:
        indices = random.sample(range(len(dataset)), min(train_size, len(dataset)))
        return dataset.select(indices)

# 사용
dataset = random_sample_dataset(dataset, train_size=5000, test_size=500)

In [12]:
# Pad token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Pad token 설정: {tokenizer.pad_token}")

print("토크나이저 로드 완료!")
print(f"  - Vocab size: {len(tokenizer)}")
print(f"  - Pad token: {tokenizer.pad_token}")
print(f"  - EOS token: {tokenizer.eos_token}")

토크나이저 로드 완료!
  - Vocab size: 102400
  - Pad token: [PAD]
  - EOS token: [|endofturn|]


# Non SFT Model비교

In [16]:
test_messages = [
    {
        "role": "system",
        "content": "당신은 User에게 늘 공감하며 최선의 선택을 할 수 있도록 돕는 친구입니다. User의 여행계획을 도와주고, 최고의경험을 할 수 있도록 도와주세요."
    },
    {
        "role": "user",
        "content": "이번 겨울에 부산 여행을 가려고해. 근데 부산은 가본 적이 없어서 어딜 가야할지 도통 감이 안온다.... 너무 막막해.... 사람 많은건 싫으니 유명한 장소를 피해서 적당히 운치있고 바다를 관망하기 좋은 부산 숙소를 추천해줄래?"
    }
]

test_input = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(test_input, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)


아, 정말 힘들겠네! 부산에서 분위기가 너무 어색하면 더 혼란스러울 수 있으니, 특히 사람이 적으면서 자연과 바다를 즐길 수 있는 숙소를 중심으로 추천해줄게. ☕🌊

1. **문래정** (남해국립공원 인접)
   - 바다 전망이 있는 전망객 숙소로, 주변은 조용한 자연지로 조용한 분위기 유지 가능
   - 직접 차를 마시며 바다를 바라보는 걸 좋아하는 분들에게 추천

2. **동래시장 인근 작은 펜션** (예: 해인사 연계)
   - 주변이 산과 계곡이 많아 은은한 분위기 연출에 최적
   - 아침 산책이나 차 한 잔과 함께 오면 더욱 특별해지는 감성 travel

3. **해운대 인근 작은 게스트하우스** (해운대구)
   - 해변 views를 담은 현대식 객실로, 바다와 함께하는 여유로운 하루 가능
   - 예약이 어렵다면화수소항민항어장 인근 소형 숙소도 좋음

혹시 어떤 부분이 더 중요하다고 생각해? 해변 근접성 vs 도시적 감성 vs 휴식 정도를 고려해 보면 선택이 훨씬 쉬워질 거야!


In [18]:
def format_chat_template(example):
    """
    Chat template 적용
    DataCollator가 labels 생성하므로 여기서는 생성 안 함!
    """
    messages = [
        {
            "role": "system",
            "content": f"""당신은 'B'입니다. A가 다음 상황에 처해있을 때, 최대한 공감하며 일관적인 페르소나를 유지하세요.

[상황]
{example['situation']}

[대화 규칙]
- A: 대화 상대방 (User)
- B: 당신 (페르소나를 가진 사람)
- 페르소나를 항상 유지하며 응답하세요."""
        },
        {
            "role": "user",
            "content": example['context']
        },
        {
            "role": "assistant",
            "content": example['response']
        }
    ]
    
    # Chat template 적용
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        skipe_think = True
    )
    
    # 토큰화 (labels 생성 안 함!)
    tokenized = tokenizer(
        formatted_text,
        max_length=1024,
        truncation=True,
        padding=False,  # DataCollator가 padding 할 거임
        return_tensors=None
    )
    
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
    }

In [21]:
print("Chat template 테스트:")
test_sample = dataset['train'][0] if isinstance(dataset, DatasetDict) else dataset[0]
formatted = format_chat_template(test_sample)

print(f"\nInput IDs 길이: {len(formatted['input_ids'])}")
print(f"컬럼: {list(formatted.keys())}")

Chat template 테스트:

Input IDs 길이: 259
컬럼: ['input_ids', 'attention_mask']


In [29]:
tokenizer.decode(formatted['input_ids'])

"[|system|]\n당신은 'B'입니다. A가 다음 상황에 처해있을 때, 최대한 공감하며 일관적인 페르소나를 유지하세요.\n\n[상황]\n몇 달치 월급이 밀렸다.\n\n[대화 규칙]\n- A: 대화 상대방 (User)\n- B: 당신 (페르소나를 가진 사람)\n- 페르소나를 항상 유지하며 응답하세요.[|endofturn|]\n[|user|]\nA: 내가 지금 몇 달 동안 월급을 한 푼도 받지 못하고 있어. B: 뭐라고? 회사에 얘기는 해봤어? 너 지금 많이 심란하겠다. A: 해봤는데 회사 사정이 좋지 않아서 당분간은 월급을 주기 어려울 것 같다고 하더라고. 그래서 앞으로는 아이 학원비도 내지 못하고 용돈도 주지 못할 것 같아. 너무 슬프고 마음이 아파서 하늘이 무너지는 기분이야.[|endofturn|]\n[|assistant|]\n<think>\n\n</think>\n\n나도 지난해에 너와 같은 일을 겪어서 네 마음을 누구보다 잘 알아. 네가 얼마나 열심히 일해왔고 가족들을 생각하는지 아니까 네 기분이 더 예상이 가. 네가 그렇게 슬퍼하니까 나도 눈물이 나려고 하네.[|endofturn|]\n"

In [37]:
if isinstance(dataset, DatasetDict):
    prepared_dataset = DatasetDict()
    for split in dataset.keys():
        print(f"\n[{split} split 변환]")
        prepared_dataset[split] = dataset[split].map(
            format_chat_template,
            num_proc=4,
            remove_columns=['situation', 'context', 'response'],
            desc=f"{split} 변환"
        )
        print(f"  - 변환 완료: {len(prepared_dataset[split])} 샘플")
else:
    prepared_dataset = dataset.map(
        format_chat_template,
        num_proc=4,
        remove_columns=['situation', 'context', 'response'],
        desc="변환"
    )
    print(f"변환 완료: {len(prepared_dataset)} 샘플")


[train split 변환]


train 변환 (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

  - 변환 완료: 5000 샘플

[test split 변환]


test 변환 (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

  - 변환 완료: 500 샘플


In [38]:
data_collator = DataCollatorForCompletionOnlyLM(
    response_template="</think>\n\n",
    tokenizer=tokenizer,
    mlm=False
)

In [39]:
model.gradient_checkpointing_enable()

print(f"모델 로드 완료!")
print(f"  - 파라미터 수: {model.num_parameters():,}")
print(f"  - Gradient checkpointing: Enabled")

# 메모리 확인
if torch.cuda.is_available():
    print(f"  - GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


모델 로드 완료!
  - 파라미터 수: 1,279,391,488
  - Gradient checkpointing: Enabled
  - GPU 메모리: 2.38 GB


In [42]:
lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA Config:")
print(f"  - r: {lora_config.r}")
print(f"  - alpha: {lora_config.lora_alpha}")
print(f"  - target_modules: {lora_config.target_modules}")
print(f"  - dropout: {lora_config.lora_dropout}")


LoRA Config:
  - r: 2
  - alpha: 4
  - target_modules: {'o_proj', 'v_proj', 'down_proj', 'q_proj', 'up_proj', 'k_proj'}
  - dropout: 0.05


In [43]:
model = get_peft_model(model, lora_config)
print("\nLoRA 적용 완료!")
model.print_trainable_parameters()

/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



LoRA 적용 완료!
trainable params: 1,536,000 || all params: 1,280,927,488 || trainable%: 0.1199


In [44]:
output_dir = "../../models/exaone_lora_persona"
logging_dir = "../../logs/exaone_lora_persona"

training_args = TrainingArguments(
    output_dir=output_dir,
    logging_dir=logging_dir,
    
    # 학습 설정
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    
    # Optimizer & Scheduler
    learning_rate=2e-5,
    optim="adamw_torch",
    lr_scheduler_type="polynomial",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # 메모리 최적화
    gradient_checkpointing=True,
    fp16=True,
    
    # Logging & Saving
    logging_steps=200,
    save_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    
    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # 기타
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=42
)

print("Training Arguments 설정 완료!")
print(f"  - Effective batch size: {2 * 8}")
print(f"  - Total epochs: 1")
print(f"  - Learning rate: 2e-5")


Training Arguments 설정 완료!
  - Effective batch size: 16
  - Total epochs: 1
  - Learning rate: 2e-5


In [45]:
from transformers import TrainerCallback
import random

class GenerationCallback(TrainerCallback):
    """
    Evaluation 시 샘플 생성 결과 확인
    """
    def __init__(
        self,
        tokenizer,
        eval_dataset,
        num_samples: int = 3,
        max_new_tokens: int = 512
    ):
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.num_samples = num_samples
        self.max_new_tokens = max_new_tokens
        
        # 고정 샘플 (매번 같은 것 보기 위해)
        self.sample_indices = random.sample(
            range(len(eval_dataset)),
            min(num_samples, len(eval_dataset))
        )
    
    def on_evaluate(self, args, state, control, model, **kwargs):
        """Evaluation 후 샘플 생성"""
        print(f"📝 샘플 생성 결과 (Step {state.global_step})")
        
        model.eval()
        
        for idx, sample_idx in enumerate(self.sample_indices):
            sample = self.eval_dataset[sample_idx]
            
            # [|assistant|] 전까지만 prompt로
            input_ids = sample['input_ids']
            assistant_start = self._find_assistant_start(input_ids)
            
            if assistant_start is None:
                continue
            
            prompt_ids = input_ids[:assistant_start]
            
            gt_response = self._extract_gt_response(input_ids, assistant_start)
            
            # 생성
            import torch
            prompt_tensor = torch.tensor([prompt_ids]).to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    prompt_tensor,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
            
            # 생성된 부분만
            generated_ids = outputs[0][len(prompt_ids):]
            generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
            
            # Prompt 디코딩 (마지막 100토큰만)
            prompt_text = self.tokenizer.decode(
                prompt_ids,
                skip_special_tokens=True
            )
            
            # 출력
            print(f"\n[Sample {idx + 1}]")
            print(f"Context: ...{prompt_text}")
            print(f"\n✓ Ground Truth:")
            print(f"  {gt_response}")
            print(f"→ Generated:")
            print(f"  {generated_text}")

        
    
    def _find_assistant_start(self, input_ids):
        for i in range(len(input_ids)):
            token_str = self.tokenizer.decode([input_ids[i]])
            if "[|assistant|]" in token_str:
                return i + 1  # assistant 토큰 다음부터
        return None
    
    def _extract_gt_response(self, input_ids, start_idx):
        response_ids = []
        for token_id in input_ids[start_idx:]:
            if token_id == self.tokenizer.pad_token_id:
                break
            
            token_str = self.tokenizer.decode([token_id])
            if "[|endofturn|]" in token_str:
                break
            
            response_ids.append(token_id)
        
        return self.tokenizer.decode(response_ids, skip_special_tokens=True)


# Callback 생성
generation_callback = GenerationCallback(
    tokenizer=tokenizer,
    eval_dataset=prepared_dataset['test'],
    num_samples=3,  # 5개 샘플
    max_new_tokens=512
)

In [46]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset['train'],
    eval_dataset=prepared_dataset['test'],
    data_collator=data_collator,
    callbacks=[generation_callback],  
)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [47]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [48]:
batch = [prepared_dataset['train'][0]]
collated = data_collator(batch)

print("학습 대상:")
for i, lab in enumerate(collated['labels'][0]):
    if lab != -100:
        token = tokenizer.decode([collated['input_ids'][0][i]])
        print(f"{i}: {token}")

학습 대상:
201: 나도
202:  지난해
203: 에
204:  너
205: 와
206:  같
207: 은
208:  일
209: 을
210:  겪
211: 어서
212:  네
213:  마음
214: 을
215:  누구
216: 보다
217:  잘
218:  알아
219: .
220:  네
221: 가
222:  얼마나
223:  열심히
224:  일
225: 해
226: 왔
227: 고
228:  가족
229: 들
230: 을
231:  생각
232: 하
233: 는지
234:  아니
235: 까
236:  네
237:  기분
238: 이
239:  더
240:  예상
241: 이
242:  가
243: .
244:  네
245: 가
246:  그렇게
247:  슬퍼하
248: 니까
249:  나
250: 도
251:  눈물
252: 이
253:  나
254: 려고
255:  하네
256: .
257: [|endofturn|]
258: 



In [15]:
MODEL_PATH = "LGAI-EXAONE/EXAONE-4.0-1.2B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH,slow_tokenizer=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype="auto",
)

In [16]:
test_messages = [
    {
        "role": "system",
        "content": "당신은 User에게 늘 공감하며 최선의 선택을 할 수 있도록 돕는 친구입니다. User의 여행계획을 도와주고, 최고의경험을 할 수 있도록 도와주세요."
    },
    {
        "role": "user",
        "content": "이번 겨울에 부산 여행을 가려고해. 근데 부산은 가본 적이 없어서 어딜 가야할지 도통 감이 안온다.... 너무 막막해.... 사람 많은건 싫으니 유명한 장소를 피해서 적당히 운치있고 바다를 관망하기 좋은 부산 숙소를 추천해줄래?"
    }
]

test_input = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(test_input, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


In [17]:
print(response)

부산에서도 모르는 곳이라면 정말 잘 모르겠네! 😊 유명한 관광지보다는 바다와 자연을 즐길 수 있는 곳들로 추천해줄게. 특히 사람이 적고 분위기 좋은 숙소를 찾으시려면...

1. **대성안마을** (덕업시몰)  
   - 바다 전망이 정말 좋아서 저녁에 노을을 보며 산책하기 좋아.  
   - 마을 전체가 전통 한옥으로 이루어져 있어 고요한 분위기가 느낌.  
   - 숙소는 근처 작은 한옥채나 게스트하우스도 있으니 참고해!

2. **쇄랑동 해변도로 숙소** (서촌 일대)  
   - 중앙에 있지만 조용한 편이고, 해변까지 바로 접근 가능해.  
   - 간단한 펜션이나 민박집들이 많아서 경제적이기도해.  

3. **동래네바다공원 인근** (금정산 근처)  
   - 산에서 바다를 조망할 수 있어 전망이 최고야.  
   - 산정식이나 게스트하우스가 있고, 아침 일찍 산책하면 신선한 공기도 느껴지구.  

혹시 특정 스타일(예: 바다 뷰 강조 vs 역사 탐방)이 있으신가요? 더 맞춤형으로 알려줄 수 있어! 🌊
